# Variance Estimation And Sample Size Planning

**Official MA1001B Alignment:** *5.6 variance and ratio of variances; 5.7 sample size estimation.*


## How To Use This Lesson

This notebook is designed as a guided teaching episode and interactive lab, not a passive code demonstration. To get the most out of this lesson:
1. **Read the conceptual explanations and explicit links** before running any code.
2. **Execute code cells sequentially**, paying attention to inline educational comments.
3. **Pause at the Guided Checkpoint** to discuss with a partner and write your reasoning before checking solutions.
4. **Complete the Independent Practice and Exit Ticket**; written justification is the primary evidence of statistical competence.


## Learning Goals

By the end of this lesson, you will be able to:
- Calculate and compare sample variances and standard deviations across operational segments.
- Compute variance ratios (`s2_B / s2_A`) to evaluate homogeneity of variance assumptions.
- Apply sample size planning formulas (`n = (z*sigma / E)^2`) to achieve targeted margins of error.
- Evaluate the quadratic cost trade-off between statistical precision and data collection expense.


## The Three Explicit Links

In accordance with the MA1001B pedagogical framework, this lesson explicitly connects theory, computation, and action:

- **1. Conceptual Link (What is modeled):** We model process variability and planning precision to determine how much empirical evidence is required.
- **2. Computational Link (How Python represents it):** We use Pandas aggregation (`.var`, `.std`) and vectorized NumPy power calculations (`np.ceil`) for sample size planning.
- **3. Decision Link (How it guides action):** Sample size planning ensures data collection budgets are spent efficiently without collecting under-powered or wasteful samples.


## Decision Scenario

> **The Problem:** An operations manager wants a precise estimate of cycle time. More precision requires more observations, which costs time and money.


## Conceptual Explanation

Variance describes variability, not error. Sample size planning connects statistical precision to operational cost. Before collecting data, analysts should decide what margin of error is useful enough for the decision.


## Mathematical Anchor

For estimating a mean, a planning approximation is n = (z* sigma / E)^2, where E is the target margin of error.


## Data And Workflow Notes

Uses simulated process data for two production lines.


## Practical Python Workflow

The following worked example demonstrates how to implement these statistical concepts in Python to generate evidence for decision making.


### Step 1: Production Line Data Simulation

We simulate cycle-time measurements across two manufacturing lines (Line A and Line B, n=90 each) having different underlying variances.


In [ ]:
# Import required data science and statistical libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set reproducible random seed and visual styling
rng = np.random.default_rng(1001)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

# Simulate cycle times for Line A (low variance) and Line B (high variance)
process = pd.DataFrame({
    "line": np.repeat(["Line_A", "Line_B"], 90),
    "cycle_time": np.r_[rng.normal(loc=10.0, scale=1.1, size=90), rng.normal(loc=10.4, scale=1.8, size=90)],
})

# Summarize count, mean, standard deviation, and variance by line
process.groupby("line")["cycle_time"].agg(
    count="count", mean="mean", std="std", variance="var"
).round(3)


### Step 2: Evaluating Variance Ratios

We isolate the cycle times for both lines and compute the ratio of variances (`Var_B / Var_A`) to quantify operational dispersion differences.


In [ ]:
# Compute sample variances and variance ratio
line_a = process.loc[process["line"].eq("Line_A"), "cycle_time"]
line_b = process.loc[process["line"].eq("Line_B"), "cycle_time"]
var_ratio = line_b.var(ddof=1) / line_a.var(ddof=1)

pd.Series({
    "variance_Line_A": line_a.var(ddof=1),
    "variance_Line_B": line_b.var(ddof=1),
    "variance_ratio_(B_to_A)": var_ratio,
    "is_variance_more_than_double": var_ratio > 2.0
}).round(3)


### Step 3: Sample Size Planning for Targeted Precision

Using the pooled standard deviation estimate, we calculate the required sample size `n` needed to achieve margins of error of 0.50, 0.25, and 0.10 units at 95% confidence (`z=1.96`).


In [ ]:
# Calculate required sample size across tightening margins of error E
est_sigma = process["cycle_time"].std(ddof=1)
planning = pd.DataFrame({"target_margin_of_error_E": [0.50, 0.25, 0.10, 0.05]})

# Apply planning formula: n = ceil( (z * sigma / E)^2 )
planning["required_sample_size_n"] = np.ceil((1.96 * est_sigma / planning["target_margin_of_error_E"]) ** 2).astype(int)
planning["relative_data_cost_multiplier"] = (planning["required_sample_size_n"] / planning["required_sample_size_n"].iloc[0]).round(1)
planning


### Step 4: Visualizing Operational Variability

We plot side-by-side boxplots of cycle times by production line to visually contrast spread, interquartile ranges, and operational stability.


In [ ]:
# Plot cycle time distributions across production lines
ax = sns.boxplot(data=process, x="line", y="cycle_time", palette="Set2", width=0.4)
ax.set_title("Cycle-Time Variability by Production Line", fontsize=14, pad=10)
ax.set_xlabel("Production Line", fontsize=11)
ax.set_ylabel("Cycle Time (Minutes)", fontsize=11)
plt.show()


## Guided Checkpoint

> [!IMPORTANT]
> **Pair Discussion & Writing Prompt:**
> Why does cutting the target margin of error in half (e.g., from 0.50 to 0.25) require four times as many observations rather than twice as many?

*Write your reasoned response below before continuing:*


## Common Mistakes & Statistical Pitfalls

Avoid these frequent errors when conducting or communicating this analysis:
- **Warning:** Treating sample size planning as a purely statistical exercise while ignoring practical data collection costs and feasibility.
- **Warning:** Comparing group averages using standard tests while ignoring massive disparities in group variances.
- **Warning:** Using an arbitrary planning standard deviation (`sigma`) without empirical justification or pilot study data.


## Independent Practice

> [!TIP]
> **Your Task:**
> Choose a practical margin of error that would be meaningful for an operations manager. Compute the required sample size `n` and write a brief justification of whether collecting that much data is financially realistic.

*Use the empty code and markdown cells below to implement your analysis and justify your recommendation.*


In [ ]:
# Write your independent practice code here
# Remember to inspect your outputs and check assumptions


## Decision Interpretation Template

Use this structured format to write your defensible conclusion and recommendation:

1. **The Decision Question:** *State the practical question being answered...*
2. **The Statistical Evidence:** *Summarize key metrics, intervals, p-values, or model comparisons...*
3. **Uncertainty & Limitations:** *Identify what the data cannot prove and what assumptions were made...*
4. **Actionable Recommendation:** *Therefore, I recommend [action] because [justification]...*


## Exit Ticket

> **Reflection:** What practical operational question should be answered before deciding on a target margin of error?

*Write your brief conceptual reflection below:*
